# ML and AI Nexus 2026 final20 audit

This notebook audits the selected final20 model only. The untested five-seed candidate and all v6-derived outputs are excluded from model selection.

In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, average_precision_score

ROOT = Path.cwd()
while not (ROOT / 'train.csv').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RUN = ROOT / 'runs' / 'final20_v11'
evidence = json.loads((ROOT / 'deliverables_final' / 'final20_evidence.json').read_text())
train = pd.read_csv(ROOT / 'train.csv')
oof = pd.read_csv(RUN / 'oof.csv')
print(evidence['recipe'])
print('Public Kaggle context:', evidence['public_kaggle_score'])

In [ ]:
assert oof.patient_id.equals(train.patient_id)
assert oof.readmitted_30d.equals(train.readmitted_30d)
y = oof.readmitted_30d
p = oof.equal_cal
print({
    'log_loss': log_loss(y, p),
    'brier': brier_score_loss(y, p),
    'roc_auc': roc_auc_score(y, p),
    'average_precision': average_precision_score(y, p),
})

In [ ]:
folds = pd.read_csv(ROOT / 'deliverables_final' / 'final20_fold_metrics.csv')
display(folds)
print('Mean calibrated fold log loss:', folds.equal_cal_log_loss.mean())
print('Std calibrated fold log loss:', folds.equal_cal_log_loss.std(ddof=1))

In [ ]:
subgroup = pd.read_csv(ROOT / 'deliverables_final' / 'final20_subgroup_metrics.csv')
display(subgroup)

In [ ]:
from IPython.display import display, Image, Markdown
display(Markdown((ROOT / 'deliverables_final' / 'TRUST_CARD.md').read_text()))
for name in ['final20_calibration.png', 'final20_fold_stability.png', 'final20_threshold_tradeoff.png', 'final20_subgroup_logloss.png']:
    display(Image(filename=str(ROOT / 'deliverables_final' / 'figures_final20' / name)))

In [ ]:
sample = pd.read_csv(ROOT / 'sample_submission.csv')
submission = pd.read_csv(RUN / 'submission_equal_cal.csv')
assert submission.columns.tolist() == sample.columns.tolist()
assert submission.patient_id.equals(sample.patient_id)
assert len(submission) == len(sample) == 3000
assert np.isfinite(submission.readmitted_30d).all()
assert submission.readmitted_30d.between(0, 1).all()
print('Final20 submission validated for manual Kaggle upload.')

## Reproduction

Run `python scripts/final20_trust_card_figures.py` from the repository root to rebuild the evidence tables, figures, Trust Card, and this notebook. The frozen model campaign itself is retained under `runs/final20_v11`.